In [20]:
import panel as pn
import pandas as pd
import hvplot.pandas
import os

In [21]:
pn.extension()
path = os.path.join("data","disaster_vietnam.xlsx")
df = pd.read_excel(path)
df["Start Year"] = pd.to_datetime(df["Start Year"], format="%Y")
df['Total Deaths'] = pd.to_numeric(df['Total Deaths'], errors='coerce')
df['Total Affected'] = pd.to_numeric(df['Total Affected'], errors='coerce')

In [22]:
df.describe(include='all')

,DisNo.,Historic,Classification Key,Disaster Group,Disaster Subgroup,Disaster Type,Disaster Subtype,External IDs,Event Name,ISO,...,Reconstruction Costs ('000 US$),"Reconstruction Costs, Adjusted ('000 US$)",Insured Damage ('000 US$),"Insured Damage, Adjusted ('000 US$)",Total Damage ('000 US$),"Total Damage, Adjusted ('000 US$)",CPI,Admin Units,Entry Date,Last Update
count,335,335,335,335,335,335,335,39,144,335,...,0.0,0.0,3.000000,3.000000,1.430000e+02,1.430000e+02,331.000000,162,335,335
unique,335,2,34,2,7,19,34,38,139,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,152,205,4
top,1953-0019-VNM,No,nat-met-sto-tro,Natural,Meteorological,Storm,Tropical cyclone,GLIDE:FL-2010-000194,Dengue,VNM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[{""adm1_code"":3328,""adm1_name"":""Bac Kan""},{""ad...",2003-07-01,2023-09-25
freq,1,223,109,261,133,133,109,2,3,335,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,99,323
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,15000.000000,24039.333333,1.636328e+05,2.235033e+05,64.472637,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,2000.000000,2549.000000,0.000000e+00,0.000000e+00,9.156133,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3500.000000,5832.500000,5.730000e+03,8.583000e+03,53.595561,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,5000.000000,9116.000000,2.300000e+04,3.824000e+04,66.731058,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,21500.000000,34784.500000,1.340000e+05,1.873700e+05,79.601309,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,38000.000000,60453.000000,6.750000e+06,8.334508e+06,100.000000,NaN,NaN,NaN


In [23]:
total = df.groupby('Disaster Type')[['Total Deaths', 'Total Affected']].sum()
print(total)
print(total.count())


                                  Total Deaths  Total Affected
Disaster Type                                                 
Air                                      269.0             5.0
Collapse (Industrial)                    453.0            80.0
Collapse (Miscellaneous)                  29.0             0.0
Drought                                    0.0       8545558.0
Epidemic                                1167.0        107907.0
Explosion (Industrial)                   114.0          4600.0
Explosion (Miscellaneous)                 47.0           559.0
Fire (Industrial)                         18.0          3521.0
Fire (Miscellaneous)                     177.0          5138.0
Flood                                   6073.0      33501230.0
Infestation                                0.0             0.0
Mass movement (wet)                      330.0         39074.0
Miscellaneous accident (General)          36.0             9.0
Poisoning                                177.0         

In [24]:
def filter_data(start_year, end_year):
    mask = (df["Start Year"].dt.year >= start_year)&(df["Start Year"].dt.year <= end_year)
    return df.loc[mask]
def create_plots(year_range):
    filtered_df = filter_data(year_range[0], year_range[1])
    time_series = filtered_df.hvplot.line(
        x = 'Start Year',
        y = 'Total Deaths',
        title = 'Total Deaths from Disaster VietNam',
        height = 300,
        responsive = True,
    )
    disaster_types = filtered_df['Disaster Type'].value_counts().hvplot.bar(
        title = 'Frequency of Disaster Types',
        height = 300,
        responsive = True,
        rot = 45
    )
    scatter_plot = filtered_df.hvplot.scatter(
        x = 'Total Deaths',
        y = 'Total Affected',
        by ='Disaster Type',
        title = 'Total Death & Total Affected from Disaster Types',
        height = 300,
        responsive = True,
    )
    return pn.Column(
        time_series,
        disaster_types,
        scatter_plot,
    )


In [25]:
year_min = df["Start Year"].dt.year.min()
year_max = df["Start Year"].dt.year.max()

year_range = pn.widgets.IntRangeSlider(
    name = 'Year Range',
    start = year_min,
    end = year_max,
    value = (year_min, year_max),
    step = 1,
)
layout = pn.Column(
    pn.pane.Markdown('Natural Disaster VietNam 1900- 2024'),
    pn.Row(year_range, align ='center'),
    pn.bind(create_plots, year_range),
    width = 900
)
layout.servable()
pn.serve(layout)


Launching server at http://localhost:64554
